# 1 - Mise à jour de la base de données

Ce notebook illustre les fonctionnalités des modules `updater` et `deleter` permettant de mettre à jour et de supprimer des données dans une base DuckLake. Les connexions sont créées via `DuckLakeConnector` et passées aux classes `DatabaseUpdater` et `DatabaseDeleter`.

## 0 - Importation des modules

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ajout du chemin vers le package parent
sys.path.append('..')

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.schema import DuckLakeTablesBuilder
from dt_ducklake_manager.operations.updater import DatabaseUpdater
from dt_ducklake_manager.operations.deleter import DatabaseDeleter

# Paramètres globaux
CATEGORICAL_THRESHOLD: int = 10
PRIMARY_KEYS: list = ['indicator', 'country', 'kind', 'model', 'training', 'week', 'horizon', 'date']
CATALOG_PATH: str = os.path.join('../outputs', 'database_update_demo.ducklake')
DATA_PATH: str    = os.path.join('../outputs', 'database_update_demo_data/')

## 1 - Construction de la base de données initiale

Construction d'un jeu de données contrôlé (~60 lignes) conçu pour illustrer les scénarios d'apparition et de disparition de tables de dimension.

**État initial visé :**
- `source` : 15 valeurs uniques > seuil=10 → **non-catégorielle**, pas de `dim_source`
- `model` : 3 valeurs uniques ≤ seuil=10 → **catégorielle**, `dim_model` créée

In [ ]:
# Initialisation du générateur aléatoire pour la reproductibilité
np.random.seed(0)

# Définition des modalités
indicators_init = ['temperature', 'humidity', 'pressure', 'wind_speed']
countries_init  = ['France', 'Germany', 'Italy', 'Spain', 'Belgium']
kinds_init      = ['forecast', 'observation']
models_init     = ['model_A', 'model_B', 'model_C']
trainings_init  = ['train_v1', 'train_v2']
horizons_init   = [1, 7, 14, 30]
weeks_init      = list(range(1, 5))

# Colonne source : 15 valeurs uniques → non-catégorielle (15 > CATEGORICAL_THRESHOLD=10)
sources_init = [f'source_{i:02d}' for i in range(1, 16)]

# Labels en français
labels_init: dict = {
    'indicator':     'Indicateur',
    'country':       'Pays',
    'kind':          'Type',
    'model':         'Modèle',
    'training':      'Entraînement',
    'week':          'Semaine',
    'horizon':       'Horizon',
    'date':          'Date',
    'value':         'Valeur',
    'lower_bound':   'Borne inférieure',
    'upper_bound':   'Borne supérieure',
    'quality_score': 'Score de qualité',
    'source':        'Source',
    'notes':         'Notes',
}

# Génération de 80 lignes (les doublons sur la clé primaire seront supprimés)
start_date_init = datetime(2024, 1, 1)
rows_init = []
for i in range(80):
    date = start_date_init + timedelta(days=i % 15)
    rows_init.append({
        'indicator':     np.random.choice(indicators_init),
        'country':       np.random.choice(countries_init),
        'kind':          np.random.choice(kinds_init),
        'model':         np.random.choice(models_init),
        'training':      np.random.choice(trainings_init),
        'week':          weeks_init[i % len(weeks_init)],
        'horizon':       horizons_init[i % len(horizons_init)],
        'date':          date,
        'value':         np.random.uniform(10, 100),
        'lower_bound':   None if np.random.random() > 0.7 else np.random.uniform(5, 50),
        'upper_bound':   None if np.random.random() > 0.7 else np.random.uniform(50, 150),
        'quality_score': np.random.uniform(0, 1),
        'source':        np.random.choice(sources_init),
        'notes':         np.random.choice(['OK', 'Warning', None], p=[0.7, 0.2, 0.1]),
    })

# Création et dédoublonnage du DataFrame sur la clé primaire composite
df_demo = pd.DataFrame(rows_init)
df_demo['date'] = pd.to_datetime(df_demo['date'])
df_demo = df_demo.drop_duplicates(subset=PRIMARY_KEYS, keep='first').reset_index(drop=True)

# Vérification des contraintes de cardinalité
assert df_demo['source'].nunique() > CATEGORICAL_THRESHOLD, (
    f"source doit avoir > {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['source'].nunique()})"
)
assert df_demo['model'].nunique() <= CATEGORICAL_THRESHOLD, (
    f"model doit avoir ≤ {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['model'].nunique()})"
)

# Affichage
print(f"Lignes après dédoublonnage   : {len(df_demo)}")
print(f"source — valeurs uniques     : {df_demo['source'].nunique()} (non-catégoriel)")
print(f"model  — valeurs uniques     : {df_demo['model'].nunique()}  (catégoriel)")
df_demo.head()

In [ ]:
# Suppression du catalogue et des données existants pour garantir un état initial propre
for suffix in ['', '.wal']:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
    print(f"Répertoire supprimé : {DATA_PATH}")

# Création de la connexion DuckLake et construction du schéma initial
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
builder = DuckLakeTablesBuilder(
    df=df_demo,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=PRIMARY_KEYS,
    connection=conn,
)
builder.build_schema(column_labels=labels_init)
conn.close()

print(f"\nBase de données créée : {CATALOG_PATH}")

In [ ]:
# Vérification de l'état initial : statut catégoriel dans les métadonnées
conn_check = DuckLakeConnector(CATALOG_PATH, DATA_PATH, read_only=True).connect()

# Affichage des méta-données
print("=== État initial — Métadonnées ===")
display(conn_check.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

# Affichage des tables de dimension
print("=== État initial — Tables de dimension ===")
display(conn_check.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

# Affichage du nombre de lignes dans la table des faits
print(f"Lignes dans fact_table : {conn_check.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")
conn_check.close()

## 2 - Mise à jour de la base de données (`DatabaseUpdater`)

Quatre scénarios sont illustrés :

1. **Scénario 2.1** : mise à jour neutre (valeurs numériques seulement) — aucun changement de tables de dimension
2. **Scénario 2.2** : l'upsert réduit les modalités de `source` à 5 valeurs (≤ seuil=10) → `dim_source` **créée**
3. **Scénario 2.3** : l'upsert introduit 12 valeurs pour `model` (> seuil=10) → `dim_model` **supprimée**
4. **Scénario 2.4** : même mise à jour que 2.1, mais avec un DataFrame **polars** — compatibilité multi-backend narwhals

In [ ]:
# Création de la connexion DuckLake et initialisation de l'updater
# La connexion est partagée avec le DatabaseDeleter en Section 3
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
updater = DatabaseUpdater(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    enable_validation=True,
)

print(f"Lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

### Scénario 2.1 — Mise à jour sans changement de tables de dimension

Mise à jour de quelques lignes existantes en modifiant uniquement les colonnes numériques (`value`, `quality_score`, `lower_bound`, `upper_bound`) et textuelles non-catégorielles (`notes`).

Aucune colonne catégorielle ni la colonne `source` ne sont modifiées, donc aucune table de dimension n'est créée ni supprimée.

> **Note technique** : `fact_table` stocke des IDs numériques pour les colonnes catégorielles. La reconstruction du `update_df` nécessite une jointure avec chaque table de dimension pour retrouver les labels humains attendus par `_prepare_dataframe_for_fact_table`.

In [ ]:
# Reconstruction des labels pour 3 lignes existantes via jointures avec les tables de dimension
sample_21 = conn.execute("""
    SELECT
        i.label  AS indicator,
        c.label  AS country,
        k.label  AS kind,
        m.label  AS model,
        t.label  AS training,
        f.week,
        f.horizon,
        f.date,
        f.source
    FROM fact_table f
    JOIN dim_indicator i ON f.indicator = i.value
    JOIN dim_country   c ON f.country   = c.value
    JOIN dim_kind      k ON f.kind      = k.value
    JOIN dim_model     m ON f.model     = m.value
    JOIN dim_training  t ON f.training  = t.value
    LIMIT 3
""").fetchdf()

# Modification des colonnes numériques uniquement
update_21 = sample_21.copy()
update_21['value']         = [99.0, 88.0, 77.0]
update_21['quality_score'] = [0.95, 0.85, 0.75]
update_21['lower_bound']   = [5.0, 4.0, 3.0]
update_21['upper_bound']   = [199.0, 180.0, 155.0]
update_21['notes']         = ['OK', 'OK', 'Warning']

print("DataFrame de mise à jour (scénario 2.1) :")
display(update_21)

In [ ]:
# Vérification de l'état AVANT la mise à jour 2.1
print("=== AVANT la mise à jour 2.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== AVANT la mise à jour 2.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# Exécution de la mise à jour 2.1
success_21 = updater.update_database(
    update_df=update_21,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.1 : {'✓ succès' if success_21 else '✗ échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.1
# Résultat attendu : métadonnées et tables de dimension inchangées
print("=== APRÈS la mise à jour 2.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

### Scénario 2.2 — Upsert créant une table de dimension (`source` → catégorielle)

L'intégralité des lignes de `fact_table` est mise à jour avec 5 nouvelles valeurs pour `source` au lieu de 15. Comme 5 ≤ seuil=10, `_update_dimensions_safe()` détecte que la colonne non-catégorielle `source` franchit désormais le seuil et appelle `dimension_mgr.convert_to_categorical('source', values)`, ce qui :
1. Crée `dim_source` avec 5 entrées
2. Met à jour `metadata.source.is_categorical` à `True`
3. Convertit les valeurs de `fact_table.source` en IDs numériques

In [ ]:
# Reconstruction de TOUTES les lignes avec les labels des colonnes catégorielles
# Nécessaire car fact_table stocke des IDs numériques pour les colonnes catégorielles
all_rows_22 = conn.execute("""
    SELECT
        i.label  AS indicator,
        c.label  AS country,
        k.label  AS kind,
        m.label  AS model,
        t.label  AS training,
        f.week,
        f.horizon,
        f.date,
        f.value,
        f.lower_bound,
        f.upper_bound,
        f.quality_score,
        f.source,
        n.label  AS notes
    FROM fact_table f
    JOIN dim_indicator i ON f.indicator = i.value
    JOIN dim_country   c ON f.country   = c.value
    JOIN dim_kind      k ON f.kind      = k.value
    JOIN dim_model     m ON f.model     = m.value
    JOIN dim_training  t ON f.training  = t.value
    LEFT JOIN dim_notes n ON f.notes    = n.value
""").fetchdf()

# Remplacement de source par 5 nouvelles valeurs (couverture uniforme sur toutes les lignes)
sources_new = ['alpha', 'beta', 'gamma', 'delta', 'epsilon']
all_rows_22['source'] = [sources_new[i % len(sources_new)] for i in range(len(all_rows_22))]

# Affichage
print(f"Lignes dans update_df          : {len(all_rows_22)}")
print(f"Valeurs uniques de source      : {sorted(all_rows_22['source'].unique())}")
print(f"Nombre de valeurs uniques      : {all_rows_22['source'].nunique()} (≤ seuil={CATEGORICAL_THRESHOLD} → conversion attendue)")

In [ ]:
# Vérification de l'état AVANT la mise à jour 2.2
print("=== AVANT la mise à jour 2.2 ===")
print("Statut de source dans les métadonnées :")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata WHERE name = 'source'"
).fetchdf())

print("Tables de dimension (dim_source absente) :")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# Exécution de la mise à jour 2.2
success_22 = updater.update_database(
    update_df=all_rows_22,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.2 : {'✓ succès' if success_22 else '✗ échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.2
# Résultat attendu : source is_categorical=True, dim_source créée avec 5 entrées
print("=== APRÈS la mise à jour 2.2 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.2 — Tables de dimension (dim_source créée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# Contenu de la nouvelle table dim_source
print("Contenu de dim_source :")
display(conn.execute("SELECT * FROM dim_source ORDER BY value").fetchdf())

# Vérification : fact_table.source contient désormais des IDs numériques
print("\nValeurs distinctes de source dans fact_table (IDs après conversion) :")
display(conn.execute("SELECT DISTINCT source FROM fact_table ORDER BY source").fetchdf())

### Scénario 2.3 — Upsert supprimant une table de dimension (`model` → non-catégorielle)

Insertion de 12 nouvelles lignes dont la colonne `model` contient 12 valeurs uniques (model_A à model_L). Comme 12 > seuil=10, `_update_dimensions_safe()` détecte que la colonne catégorielle `model` dépasse le seuil et appelle `dimension_mgr.convert_to_non_categorical('model')`, ce qui :
1. Convertit les IDs dans `fact_table.model` en labels VARCHAR
2. Supprime `dim_model`
3. Met à jour `metadata.model.is_categorical` à `False`

> **Note** : après le scénario 2.2, `source` est catégorielle avec `dim_source = {alpha, beta, gamma, delta, epsilon}`. Les nouvelles lignes doivent utiliser un label existant pour `source`.

In [ ]:
# Construction de 12 nouvelles lignes, une par valeur de model (model_A à model_L)
# Combinaison de clé primaire fixe pour éviter tout conflit avec les lignes existantes
all_models = [f'model_{chr(65 + i)}' for i in range(12)]  # model_A ... model_L

rows_23 = []
for model in all_models:
    rows_23.append({
        'indicator':     'temperature',
        'country':       'France',
        'kind':          'forecast',
        'model':         model,
        'training':      'train_v1',
        'week':          52,
        'horizon':       30,
        'date':          pd.Timestamp('2025-01-01'),
        'source':        'alpha',    # label existant dans dim_source (catégorielle après 2.2)
        'value':         np.random.uniform(10, 100),
        'lower_bound':   None,
        'upper_bound':   None,
        'quality_score': np.random.uniform(0, 1),
        'notes':         'OK',
    })

update_23 = pd.DataFrame(rows_23)

# Affichage
print(f"Lignes dans update_df          : {len(update_23)}")
print(f"Valeurs uniques de model       : {sorted(update_23['model'].unique())}")
print(f"Nombre de valeurs uniques      : {update_23['model'].nunique()} (> seuil={CATEGORICAL_THRESHOLD} → suppression attendue de dim_model)")

In [ ]:
# Vérification de l'état AVANT la mise à jour 2.3
print("=== AVANT la mise à jour 2.3 ===")
print("Statut de model dans les métadonnées :")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata WHERE name = 'model'"
).fetchdf())

print("Contenu actuel de dim_model :")
display(conn.execute("SELECT * FROM dim_model ORDER BY value").fetchdf())

In [ ]:
# Exécution de la mise à jour 2.3
success_23 = updater.update_database(
    update_df=update_23,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.3 : {'✓ succès' if success_23 else '✗ échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.3
# Résultat attendu : model is_categorical=False, dim_model supprimée
print("=== APRÈS la mise à jour 2.3 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.3 — Tables de dimension (dim_model supprimée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# Vérification : fact_table.model contient désormais des labels VARCHAR (pas des IDs)
print("Valeurs distinctes de model dans fact_table (labels VARCHAR après conversion) :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

### Scénario 2.4 — Compatibilité multi-backend narwhals (DataFrame polars)

`update_database()` est typé `IntoDataFrame` et convertit l'entrée en narwhals dès le point d'entrée.
Il accepte donc indifféremment un DataFrame **pandas** ou **polars** (ou tout autre backend narwhals-compatible).

Cet exemple reproduit le scénario 2.1 avec un DataFrame **polars** : seules des colonnes numériques
sont modifiées, les métadonnées et tables de dimension restent inchangées.

In [ ]:
import polars as pl

# Conversion du DataFrame pandas update_21 en polars
update_24_polars = pl.from_pandas(update_21)
print(f"Type de l'entrée : {type(update_24_polars).__name__}")
display(update_24_polars)

In [ ]:
# Exécution de la mise à jour 2.4 avec un DataFrame polars
success_24 = updater.update_database(
    update_df=update_24_polars,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.4 (polars backend) : {'✓ succès' if success_24 else '✗ échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.4
# Résultat attendu : identique à l'état après 2.1 — métadonnées et tables de dimension inchangées
print("=== APRÈS la mise à jour 2.4 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.4 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

## 3 - Suppression de données (`DatabaseDeleter`)

Deux scénarios sont illustrés :

1. **Scénario 3.1** : suppression des lignes de la Belgique — aucun changement de tables de dimension (model conserve 12 valeurs uniques > seuil)
2. **Scénario 3.2** : suppression des lignes `model_D` à `model_L` — `model` retrouve 3 valeurs uniques ≤ seuil=10 → `dim_model` **recréée**

In [ ]:
# Initialisation du deleter en partageant la connexion ouverte par l'updater
# Les deux classes opèrent sur le même état de la base de données
deleter = DatabaseDeleter(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    enable_validation=True,
    auto_cleanup=True,
)

print(f"Lignes dans fact_table avant suppressions : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

### Scénario 3.1 — Suppression sans changement de tables de dimension

Suppression de toutes les lignes correspondant à la Belgique.

Deux effets attendus après la suppression :

1. **`Belgium` disparaît de `dim_country`** — la suppression des lignes de fait rend l'entrée orpheline ;
   `perform_cleanup=True` déclenche `cleanup_orphaned_dimension_entries()` qui retire toute entrée de
   table de dimension non référencée dans `fact_table`.
2. **`model` conserve ses 12 valeurs uniques** — les 12 lignes insérées en 2.3 utilisent toutes
   `country='France'` ; la suppression des lignes belges ne les affecte pas. Le nombre de modalités
   distinctes de `model` dans `fact_table` reste donc supérieur au seuil → aucune conversion.

> **Note** : `country` est catégorielle → le filtre utilise l'ID numérique stocké dans `fact_table`, récupéré depuis `dim_country`.

In [ ]:
# Récupération de l'identifiant de Belgium dans dim_country
# country est catégorielle → fact_table stocke des IDs → le filtre doit utiliser l'ID
belgium_id = conn.execute(
    "SELECT value FROM dim_country WHERE label = 'Belgium'"
).fetchone()[0]

print(f"ID de Belgium dans dim_country : {belgium_id!r}")

# Comptage préalable des lignes à supprimer
n_belgium = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE country = '{belgium_id}'"
).fetchone()[0]
print(f"Lignes à supprimer             : {n_belgium}")

In [ ]:
# Vérification de l'état AVANT la suppression 3.1
print("=== AVANT la suppression 3.1 — dim_country ===")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

print(f"Valeurs uniques de country : {conn.execute('SELECT COUNT(DISTINCT country) FROM fact_table').fetchone()[0]}")

print(f"Valeurs uniques de model : {conn.execute('SELECT COUNT(DISTINCT model) FROM fact_table').fetchone()[0]}")

In [ ]:
# Exécution de la suppression 3.1
filters_31 = [('country', '=', belgium_id)]

deleted_31 = deleter.delete_rows(
    filters=filters_31,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_31}")

In [ ]:
# Vérification de l'état APRÈS la suppression 3.1
# Résultat attendu : Belgium absent de dim_country, aucune table créée/supprimée
print("=== APRÈS la suppression 3.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la suppression 3.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# Belgium doit avoir disparu de dim_country (nettoyage des orphelins déclenché par perform_cleanup=True)
print("Contenu de dim_country après suppression :")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

# country possède désormais 4 valeurs uniques
unique_country_count = conn.execute(
    "SELECT COUNT(DISTINCT country) FROM fact_table"
).fetchone()[0]
print(f"Valeurs uniques de country après suppression Belgique : {unique_country_count}")

# model conserve 12 valeurs uniques : les lignes France de 2.3 (avec model_A...model_L) sont intactes
unique_model_count = conn.execute(
    "SELECT COUNT(DISTINCT model) FROM fact_table"
).fetchone()[0]
print(f"Valeurs uniques de model après suppression Belgique : {unique_model_count} (> seuil={CATEGORICAL_THRESHOLD} → pas de conversion)")

### Scénario 3.2 — Suppression créant une nouvelle table de dimension (`model` → catégorielle)

Suppression des lignes dont le modèle appartient à `{model_D, ..., model_L}`. Après la suppression, seuls `model_A`, `model_B`, `model_C` subsistent (3 valeurs ≤ seuil=10).

Le nettoyage post-suppression (`perform_cleanup=True`) déclenche `_detect_new_categorical_after_deletion()` qui détecte ce franchissement de seuil et appelle `convert_to_categorical('model')` :
1. Crée `dim_model` avec 3 entrées
2. Convertit les labels VARCHAR de `fact_table.model` en IDs numériques
3. Met à jour `metadata.model.is_categorical` à `True`

> **Note** : `model` est non-catégorielle après le scénario 2.3 → `fact_table.model` stocke des labels VARCHAR → le filtre SQL utilise les labels directement.

In [ ]:
# Modèles à supprimer (ceux introduits en 2.3 qui dépassent le seuil)
models_to_delete = [f'model_{chr(65 + i)}' for i in range(3, 12)]  # model_D ... model_L
print(f"Modèles à supprimer : {models_to_delete}")

# Comptage préalable des lignes à supprimer
models_sql = ', '.join([f"'{m}'" for m in models_to_delete])
n_to_delete = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE model IN ({models_sql})"
).fetchone()[0]
print(f"Lignes à supprimer : {n_to_delete}")

# Filtre SQL en chaîne de caractères (model est VARCHAR non-catégorielle → labels directs)
filter_32 = f"model IN ({models_sql})"
print(f"Filtre SQL         : {filter_32}")

In [ ]:
# Vérification de l'état AVANT la suppression 3.2
print("=== AVANT la suppression 3.2 ===")
print("Valeurs distinctes de model dans fact_table :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

print(f"\nTotal lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

In [ ]:
# Exécution de la suppression 3.2
# perform_cleanup=True est essentiel pour déclencher _detect_new_categorical_after_deletion()
deleted_32 = deleter.delete_rows(
    filters=filter_32,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_32}")

In [ ]:
# Vérification de l'état APRÈS la suppression 3.2
# Résultat attendu : model is_categorical=True, dim_model recréée avec 3 entrées
print("=== APRÈS la suppression 3.2 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la suppression 3.2 — Tables de dimension (dim_model recréée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

In [ ]:
# dim_model doit être recréée avec 3 entrées
print("Contenu de dim_model (recréée) :")
display(conn.execute("SELECT * FROM dim_model ORDER BY value").fetchdf())

# fact_table.model contient à nouveau des IDs numériques
print("\nValeurs distinctes de model dans fact_table (IDs numériques après reconversion) :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

print(f"\nTotal lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")